In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
def load_dataset(filepath="raw_skills.csv"):
    """Load job roles and their associated skills from CSV."""
    df = pd.read_csv(filepath)
    print(f"✅ Dataset loaded: {len(df)} job roles found.\n")
    return df

In [3]:
def build_tfidf_matrix(df):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df["skills"])
    vocabulary = vectorizer.get_feature_names_out()
    
    print(f"✅ TF-IDF matrix built: {tfidf_matrix.shape[0]} roles × {tfidf_matrix.shape[1]} unique skills.\n")
    return vectorizer, tfidf_matrix, vocabulary

In [6]:
def get_user_profile():
   
    print("=" * 55)
    print("   🚀  TECH STACK RECOMMENDER — DecodeLabs  ")
    print("=" * 55)
    print("\nEnter your skills to get matched job roles!")
    print("(Minimum 3 skills required for accurate matching)\n")
 
    skills = []
    while len(skills) < 3:
        remaining = 3 - len(skills)
        prompt = f"  Enter skill {len(skills)+1} (at least {remaining} more needed): "
        skill = input(prompt).strip()
        if skill:
            skills.append(skill)
 
    # Allow extra skills
    print("\n  Add more skills? (press Enter to skip)")
    while True:
        skill = input(f"  Enter skill {len(skills)+1} (or press Enter to finish): ").strip()
        if not skill:
            break
        skills.append(skill)
 
    print(f"\n✅ User Profile Captured: {skills}\n")
    return skills

In [7]:
def score_recommendations(user_skills, vectorizer, tfidf_matrix, df):
    
    # Convert list to space-separated string (same format as dataset)
    user_text = " ".join(user_skills)
    
    # Transform using the fitted vectorizer (MUST use same vocabulary space)
    user_vector = vectorizer.transform([user_text])
    
    # Compute cosine similarity against all job role vectors
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()
    
    # Attach scores to the dataframe
    results = df.copy()
    results["similarity_score"] = similarity_scores
    
    return results

In [8]:
def get_top_recommendations(results, top_n=3):
    
    # Sort by similarity score (highest first)
    sorted_results = results.sort_values("similarity_score", ascending=False)
    
    # Filter to top N
    top_results = sorted_results.head(top_n)
    
    return top_results

In [9]:
def display_results(top_results, user_skills):
    
    print("=" * 55)
    print(f"   🎯  TOP {len(top_results)} RECOMMENDED CAREER PATHS")
    print("=" * 55)
    
    for rank, (_, row) in enumerate(top_results.iterrows(), start=1):
        score = row["similarity_score"]
        role  = row["job_role"]
        role_skills = set(row["skills"].replace("_", " ").lower().split())
        user_set    = set(s.lower() for s in user_skills)
        
        # Find matching skills
        matched = [s for s in user_skills if s.lower() in role_skills]
        match_pct = int(score * 100)
 
        print(f"\n  {'🥇' if rank==1 else '🥈' if rank==2 else '🥉'} Rank #{rank}: {role}")
        print(f"     Match Score : {score:.4f}  ({match_pct}% alignment)")
        print(f"     Your Skills : {', '.join(user_skills)}")
        print(f"     Matched With: {', '.join(matched) if matched else 'Partial overlap via TF-IDF'}")
        print(f"     Role Skills : {row['skills'].replace('_', ' ')}")
 
    print("\n" + "=" * 55)
    
    # Cold start warning
    if top_results["similarity_score"].max() == 0:
        print("\n⚠️  COLD START DETECTED: No skill overlap found.")
        print("   → Try entering skills from this list:")
        print("   Python, SQL, Machine_Learning, Docker, AWS,")
        print("   JavaScript, React, Security, Linux, TensorFlow")
    
    print("\n✅ Recommendation complete! Build your skills accordingly.")
    print("=" * 55)

In [10]:
def main():
    # Step 1: Ingestion
    df = load_dataset("raw_skills.csv")
    
    # Step 2: Build TF-IDF Matrix
    vectorizer, tfidf_matrix, vocabulary = build_tfidf_matrix(df)
    
    # Step 3: Get User Input (minimum 3 skills)
    user_skills = get_user_profile()
    
    # Step 4: Score all job roles using Cosine Similarity
    results = score_recommendations(user_skills, vectorizer, tfidf_matrix, df)
    
    # Step 5: Sort & Filter → Top 3
    top_results = get_top_recommendations(results, top_n=3)
    
    # Step 6: Display results
    display_results(top_results, user_skills)

In [11]:
main()

✅ Dataset loaded: 15 job roles found.

✅ TF-IDF matrix built: 15 roles × 75 unique skills.

   🚀  TECH STACK RECOMMENDER — DecodeLabs  

Enter your skills to get matched job roles!
(Minimum 3 skills required for accurate matching)



  Enter skill 1 (at least 3 more needed):  python machine learning , deep learning
  Enter skill 2 (at least 2 more needed):  machine learning
  Enter skill 3 (at least 1 more needed):  deep learning



  Add more skills? (press Enter to skip)


  Enter skill 4 (or press Enter to finish):  tensorflow
  Enter skill 5 (or press Enter to finish):  



✅ User Profile Captured: ['python machine learning , deep learning', 'machine learning', 'deep learning', 'tensorflow']

   🎯  TOP 3 RECOMMENDED CAREER PATHS

  🥇 Rank #1: ML Engineer
     Match Score : 0.3971  (39% alignment)
     Your Skills : python machine learning , deep learning, machine learning, deep learning, tensorflow
     Matched With: tensorflow
     Role Skills : Python TensorFlow PyTorch Machine Learning Deep Learning Docker Kubernetes MLOps APIs

  🥈 Rank #2: Data Scientist
     Match Score : 0.3684  (36% alignment)
     Your Skills : python machine learning , deep learning, machine learning, deep learning, tensorflow
     Matched With: tensorflow
     Role Skills : Python SQL Machine Learning Data Analysis Statistics TensorFlow Pandas NumPy Visualization

  🥉 Rank #3: AI Engineer
     Match Score : 0.3644  (36% alignment)
     Your Skills : python machine learning , deep learning, machine learning, deep learning, tensorflow
     Matched With: tensorflow
     Role Skil